In [7]:
import csv
import smtplib
import ssl
import time
import logging
from email.message import EmailMessage
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

# Logging setup
logging.basicConfig(filename="email_sender.log", level=logging.INFO)

SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 465
SENDER_EMAIL = "your_email@gmail.com"
APP_PASSWORD = "your_app_password"

MAX_RETRIES = 3
RETRY_DELAY = 5

def create_report(name, stats, filename):
    """Generate a personalized PDF report."""
    c = canvas.Canvas(filename, pagesize=letter)
    width, height = letter

    c.setFont("Helvetica-Bold", 20)
    c.drawCentredString(width / 2, height - 100, "Monthly Report")

    c.setFont("Helvetica", 14)
    c.drawString(100, height - 150, f"Prepared for: {name}")
    c.drawString(100, height - 170, "Date: March 12, 2026")

    c.setFont("Helvetica", 12)
    text = c.beginText(100, height - 220)
    text.textLines(f"""
    Dear {name},

    This is your monthly performance report.
    - Sales: {stats.get('sales', 'N/A')}
    - Leads: {stats.get('leads', 'N/A')}
    - Satisfaction: {stats.get('satisfaction', 'N/A')}

    Keep up the great work!

    Regards,
    Analytics Team
    """)
    c.drawText(text)

    c.showPage()
    c.save()

def send_email(recipient, subject, body, attachment_path):
    """Send email with attachment and retry logic."""
    msg = EmailMessage()
    msg["From"] = SENDER_EMAIL
    msg["To"] = recipient
    msg["Subject"] = subject
    msg.set_content(body)

    # Attach PDF
    with open(attachment_path, "rb") as f:
        msg.add_attachment(f.read(), maintype="application", subtype="pdf", filename=attachment_path)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            context = ssl.create_default_context()
            with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT, context=context) as server:
                server.login(SENDER_EMAIL, APP_PASSWORD)
                server.send_message(msg)
            logging.info(f"Email sent to {recipient}")
            return True
        except Exception as e:
            logging.warning(f"Attempt {attempt} failed for {recipient}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
            else:
                logging.error(f"Failed to send email to {recipient}")
                return False

def main():
    # Example CSV format: name,email,sales,leads,satisfaction
    with open("recipients.csv", newline="") as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            name = row["name"]
            email = row["email"]
            stats = {
                "sales": row.get("sales"),
                "leads": row.get("leads"),
                "satisfaction": row.get("satisfaction")
            }

            pdf_filename = f"{name}_report.pdf"
            create_report(name, stats, pdf_filename)

            subject = f"Hello {name}, here is your personalized report"
            body = f"Dear {name},\n\nPlease find attached your latest report.\n\nBest regards,\nTeam"

            send_email(email, subject, body, pdf_filename)

if __name__ == "__main__":
    main()


ERROR:root:Failed to send email to alice@example.com
ERROR:root:Failed to send email to bob@example.com
